# Datamine MINPER Process Example

This notebook demonstrates how to configure and run the **`minper`** process wrapper in `dmstudio`.

## Process Description

## MINPER

Modify a set of mining design perimeters automatically, according to production scheduling data.

**MINPER** has application in both underground and open pit mining including:-

* The generation of plans for a proposed mine design at any required time during a mining schedule.

* The production of a series of mine plans or isometric plots, depicting the mine being excavated from start to finish.

* For open pit situations, the production of mined profiles according to the proposed schedule, so that face slopes and advances can be checked at different times during the schedule.

The input perimeter file contains perimeters which correspond to the original production unit data that was used for production scheduling. The perimeters should be primarily related to plan definitions. Corresponding to this set of perimeters, a set of input mining direction strings are defined. The file with this data may contain simply one two point string, which describes a vector for use as a mining direction for all perimeters. Alternatively, separate direction strings may be defined for each mining perimeter, in which case the PVALUEs of the mining direction strings and perimeters should correspond. It does not matter whether the mining direction strings are inside or outside the perimeters, or whether they actually intersect the mining perimeters. The mining direction strings can be just two point vectors, or they may have many points, to represent a mining face which changes its orientation as it advances.

The other input file is the &SHEDRES file, which contains production scheduling data at some particular point in time. This will normally have been produced by the PRODSH process, either by saving a particular file during use of the LIST UNIT CUM BAR [/DU] command, or by copying with retrieval criteria (based on a selected time) from a complete output schedule file. It will normally contain (and needs for the operation of MINPER) the fields PNUM, SNUM, TONNES and TOTALTON. The entries in the TONNES field indicate the amount of reserves left at the time the file was created, and the entries in the TOTAL TON field indicate the tonnage that has been mined up to and including that time slot.

You must also define which field in the &**SHEDRES** file corresponds to the PVALUE entries in the perimeter file, normally either PNUM or SNUM. This depends on the way in which the original &RESERVE file was prepared for entry to PRODSH.

If the &SHEDRES file comes from sources other than PRODSH, the * **PERIM** field name entry can be anything, as long as its data entries correspond to the logical PVALUE entries in the input perimeter file.

The @**MINE** parameter can be set to 0 if the user requires, to produce the remainder of the perimeters that remain to be 'mined', rather than the already 'mined' part of the perimeters (@**MINE** =1 which is the default).

The output from **MINPER** is a set of perimeters, which obviously are related to the original perimeter shapes, but have one truncated edge, representing a mining face perpendicular to the mining direction string. The position of this edge has been determined from the proportion of the perimeter which has been 'mined'. If more than one mining direction string was present in the &STRINGIN file, then only perimeters which have a corresponding string will be output. Similarly, perimeters for which there is no corresponding schedule data will not be output.

The output set of perimeters can be used to create a plan or isometric plot of the mine design at the particular time of the schedule data. Alternatively, the strings may be used as the basis of more detailed mine design work.

### Series of Mined Excavations

After a complete schedule has been produced using PRODSH, it is possible to utilize MINPER to generate a whole set of perimeters, depicting the mining excavation at selected time slots throughout the schedule. This is done by using a looping macro, in which data for particular time slots is retrieved consecutively from the schedule, and then used with MINPER to produce the mined perimeters for that time slot. This procedure may be outlined as follows:-

1. The user must input the start time, the end time, and the time increment required.

2. Starting at the first time slot, the data from the schedule for just that time slot is copied out by retrieval.

3. The MINPER process is used to determine the 'mined' perimeters at that time.

4. Using processes [EXTRA](<extra.md>) and [PROPER](<proper.md>), the original mid-bench perimeters may be converted into the pairs of corresponding toe and crest perimeters.

5. The processes [SURTRI](<surtri.md>) and [WIREPE](<wirepe.md>) are then used to create a crossing latticework of strings over the pit surface.

6. The process [ISOPER](<isoper.md>) is finally used to create a 3D plot of the pit surface at that time in the schedule. This plot may be logically named according to the current time slot, and incorporated into a plot catalogue file.

7. The time slot is then incremented and steps (2) to (6) repeated until the end time is reached.

8. The final catalogue file may then be displayed to give the user a complete run through the proposed production schedule.

Using the wireframe model produced in step (5), section profile strings along any required section line may be produced. This may be done to produce face profile strings, which may be colored differently for the different times, and then viewed in the appropriate sectional view in the design window. Querying facilities allow the user to get detailed advance distances and slopes resulting from the proposed schedule.

### Input Files:

* **perimin** (String):
  Input perimeter file, which must contain the fields **XP** , **YP** , **ZP** , **PTN**
  and **PVALUE**.
  Required=Yes

* **shedres** (Undefined):
  Input schedule data file, which must contain at least the fields **TONNES** ,
  **TOTALTON** and another field whose entries correspond with the **PVALUE** entries in
  the **PERIMIN** file. If this file has been produced using the **PRODSH** process, it
  will contain the fields **PNAME** , **PNUM** , **SNUM** , **TONNES** , **TOTALTON** ,
  **PRATE** and a number of grade fields.
  Required=Yes

* **stringin** (String):
  Input mining string file, which defines the direction of mining. It may contain just
  one string, that will be applied to all the perimeters. Alternatively, it may contain a
  number of strings, which define different mining directions for each of the input
  perimeters. In the latter case, there must be match between the **PVALUE** entries in
  the **PERIMIN** and **STRINGIN** files.
  Required=Yes

### Output Files:

* **perimout** (String):
  Output perimeter file, which will contain the perimeters which have been modified
  according to the input schedule data.
  Required=Yes

### Fields:

* **perim** (Numeric : **PERIMIN**):
  Field name in the input **SHEDRES** file that corresponds with the **PVALUE** entries
  in the input **PERIMIN** file.
  Default=Undefined
  Required=Yes

### Parameters:

* **mine**:
  Flag indicating whether the output perimeter file should contain unmined perimeters, 0,
  or the mined perimters, (1).
  Range=0,1
  Values=0,1
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('minper')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute minper
print("Running minper...")
dm_cmd.minper(
    perimin_i='t_input_file',  # required
    shedres_i=['t_input_file'],  # required
    stringin_i='t_SurfaceTriangles',  # required
    perimout_o='t_minper_out',  # required
    perim_f='optional',  # required
    # mine_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("minper execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_minper_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")